In [7]:
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_core.messages import HumanMessage

load_dotenv()

True

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server" : {
            "transport" : "http",
            "url" : "https://mcp.kiwi.com"
        }
    }
)

tools = await client.get_tools()

In [9]:
from langchain_groq import  ChatGroq
travel_model = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

In [10]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

travel_agent = create_agent(travel_model,
                     tools=tools,
                     checkpointer=InMemorySaver(),
                     system_prompt="You are a travel agent. Your job is to know the users needs and recommend flights according to that and ask the follow up questions if required for the flight"
                     )

In [13]:

response = await travel_agent.ainvoke(
    {"messages" : [HumanMessage(content="Departure date is 25 october 2026 ")]},
    {"configurable" : {"thread_id" : "flight"}}
)

print(response["messages"][-1].content)


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-20b` in organization `org_01m1tg10h9eembspekemg34sb1` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 9606, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
@tool
def travel_specialist(query: str) -> str:
    """Delegate flight, pricing, and travel search requests to the travel specialist"""

    result = travel_agent.invoke(
        {"messages" : [HumanMessage(content=query)]},
        {"configurable" : {"thread_id" : "isolated_travel_thread"}}
    )

    return result["messages"][-1].content
